In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# Locate cleaned jobs data
jobs_path = Path("data/processed/jobs_clean.csv")
if not jobs_path.exists():
    jobs_path = Path("../data/processed/jobs_clean.csv")

jobs = pd.read_csv(jobs_path)

# Vectorize corpus for clustering
cluster_vec = TfidfVectorizer(max_features=2500, stop_words="english")
X = cluster_vec.fit_transform(jobs["text"].fillna(""))
print("Feature matrix shape:", X.shape)

Feature matrix shape: (20000, 2500)


In [ ]:
k = 10
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
jobs["cluster"] = kmeans.fit_predict(X)

order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = cluster_vec.get_feature_names_out()

print("Discovered Job Clusters:")
for i in range(k):
    top_terms = [terms[ind] for ind in order_centroids[i, :8]]
    print(f"Cluster {i}: {', '.join(top_terms)}")

Discovered Job Clusters:
Cluster 0: experience, work, team, time, position, health, skills, job
Cluster 1: store, sales, customer, customers, merchandise, retail, service, manager
Cluster 2: care, patient, nursing, patients, nurse, health, healthcare, rn
Cluster 3: data, experience, design, technical, software, engineering, engineer, systems
Cluster 4: doctorate, profile, specialization, description, role, job, send, jobs
Cluster 5: financial, business, accounting, experience, work, team, management, marketing
Cluster 6: equipment, work, safety, maintenance, ability, required, job, service
Cluster 7: sales, business, customer, marketing, customers, account, new, territory
Cluster 8: software, programming, doctorate, profile, design, specialization, developer, application
Cluster 9: project, construction, projects, engineering, management, design, experience, manager


In [ ]:
import re

# Standard skill vocabulary
TECH_SKILLS_TAXONOMY = [
    "python", "r", "sql", "machine learning", "deep learning", "nlp",
    "data analysis", "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch",
    "tableau", "power bi", "matplotlib", "seaborn", "statistics", "aws",
    "docker", "kubernetes", "git", "spark", "hadoop", "excel", "bigquery"
]

def analyze_skill_gap(resume_text: str, target_role_keyword: str) -> dict:
    # 1. Filter the matching target role
    matched_jobs = jobs[jobs["title"].str.contains(target_role_keyword, case=False, na=False)]
    
    if len(matched_jobs) == 0:
        return {"error": f"No job postings found matching '{target_role_keyword}'"}
    
    # 2. Extract skills that appear in job descriptions
    combined_job_corpus = " ".join(matched_jobs["text"].dropna().str.lower())
    
    demanded_skills = []
    for skill in TECH_SKILLS_TAXONOMY:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, combined_job_corpus):
            demanded_skills.append(skill)
            
    # 3. Check if demanded skills is in candidates resume
    resume_lower = f" {resume_text.lower()} "
    matching_skills = []
    missing_skills = []
    
    for skill in demanded_skills:
        pattern = r"\b" + re.escape(skill) + r"\b"
        if re.search(pattern, resume_lower):
            matching_skills.append(skill)
        else:
            missing_skills.append(skill)
            
    total_demanded = len(demanded_skills)
    readiness = (len(matching_skills) / max(total_demanded, 1)) * 100
    
    return {
        "target_role": target_role_keyword,
        "matching_skills": matching_skills,
        "missing_skills": missing_skills,
        "readiness_score": f"{readiness:.1f}%"
    }

# Test profile
test_resume = """
Experienced Data Scientist with expertise in Python, SQL, Pandas, 
Machine Learning, Data Analysis, and Matplotlib.
"""
gap_report = analyze_skill_gap(test_resume, target_role_keyword="Data Scientist")

print("Updated Skill Gap Report:")
for key, val in gap_report.items():
    print(f"- {key}: {val}")

Updated Skill Gap Report:
- target_role: Data Scientist
- matching_skills: ['python', 'sql', 'machine learning', 'data analysis', 'pandas', 'matplotlib']
- missing_skills: ['r', 'deep learning', 'nlp', 'numpy', 'tensorflow', 'pytorch', 'tableau', 'statistics', 'aws', 'docker', 'kubernetes', 'git', 'spark', 'hadoop', 'excel']
- readiness_score: 28.6%
